## **Stacking (Ensemble Method)**

#### **Stacking Regressor**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv('china_used_cars.csv')

drop_cols = ['price', 'mileage_km', 'log_mileage', 'mileage_per_year', 'year', 'month']
X = df.drop(columns=drop_cols)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

base_learners = [
    ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingRegressor(random_state=42)),
    ('lr', make_pipeline(StandardScaler(), LinearRegression())),
]

stack_reg = StackingRegressor(
    estimators=base_learners,
    final_estimator=RidgeCV(),   # meta-model — RidgeCV is a common, stable default
    cv=5,                        # 5-fold CV to generate honest out-of-fold predictions
    n_jobs=-1
)
stack_reg.fit(X_train, y_train)
pred = stack_reg.predict(X_test)
print("Stacking -> MAE:", mean_absolute_error(y_test, pred), " R2:", r2_score(y_test, pred))

for name, est in stack_reg.named_estimators_.items():
    p = est.predict(X_test)
    print(f"{name:4s} -> MAE: {mean_absolute_error(y_test, p):.2f}  R2: {r2_score(y_test, p):.4f}")

# Inspect what the meta-model learned — the weight given to each base learner
print("Meta-model coefficients (rf, gb, lr):", stack_reg.final_estimator_.coef_)
print("Meta-model intercept:", stack_reg.final_estimator_.intercept_)

Stacking -> MAE: 17187.31534730699  R2: 0.7057222692405889
rf   -> MAE: 12357.65  R2: 0.7044
gb   -> MAE: 19420.69  R2: 0.3796
lr   -> MAE: 34686.28  R2: 0.2235
Meta-model coefficients (rf, gb, lr): [ 1.01915583 -0.38148427  0.35658024]
Meta-model intercept: -124.4574436841649


#### **Stacking Classifer**

In [2]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

drop_cols = ['is_electric', 'battery_capacity_kwh', 'motor_power_kw',
             'price', 'mileage_km', 'log_mileage', 'mileage_per_year', 'year', 'month']
X = df.drop(columns=drop_cols)
y = df['is_electric']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

base_clf = [
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(random_state=42)),
]

stack_clf = StackingClassifier(
    estimators=base_clf,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)
stack_clf.fit(X_train, y_train)
pred = stack_clf.predict(X_test)
print("Stacking accuracy:", accuracy_score(y_test, pred))

for name, est in stack_clf.named_estimators_.items():
    p = est.predict(X_test)
    print(f"{name} accuracy: {accuracy_score(y_test, p):.4f}")

Stacking accuracy: 0.994026284348865
rf accuracy: 0.9904
gb accuracy: 0.9916
